# Example 11.6.1 / 3: Optimization KKT Systems versus MPCC

Companion notebook for **11.6.1 Equilibrium Problems and MPCC** in the AMPL lab Documentation, section "Optimization KKT Systems versus MPCC". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Optimization KKT Systems versus MPCC

KKT conditions of a lower-level optimization problem can sometimes be embedded as complementarity constraints in a single-level model. This creates an MPCC and requires all assumptions behind the KKT replacement. It is not automatically equivalent when constraint qualifications or lower-level convexity conditions are absent.

The replacement must include primal feasibility, dual feasibility, stationarity, and every
complementarity pair with a consistent sign convention. For a convex lower-level problem under
an appropriate constraint qualification, KKT conditions can characterize global lower-level
optimality. For a nonconvex lower level they are generally only necessary local conditions.

A small bilevel instance. The leader chooses $x\in[-2,2]$ to minimize $(x-1)^2+(y+1)^2$, where
$y$ solves the convex lower level $\min_y (y-x)^2$ subject to $y\ge0$, so $y=\max(x,0)$. The
lower level has a linear constraint, so its KKT conditions are exact: stationarity
$2(y-x)-\lambda=0$ and the pair $0\le y\perp\lambda\ge0$. By hand, $x\ge0$ gives $2x^2+2$ and
$x<0$ gives $(x-1)^2+1$, so the optimum is $x=y=0$ with value 2. The MPCC solved by Gurobi
agrees (this run needs a Gurobi licence; see the note at the top):

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2;  # leader decision
var y;              # follower decision
var Lambda;         # multiplier of the follower's y >= 0

minimize Upper: (x - 1)^2 + (y + 1)^2;  # leader objective

subject to Stationarity: 2 * (y - x) - Lambda = 0;      # follower KKT stationarity
subject to LowerCompl: y >= 0 complements Lambda >= 0;  # follower bound and its multiplier

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.solve(solver="gurobi", verbose=False)
print(ampl.solve_result, ampl.var["x"].value(), ampl.var["y"].value(),
      ampl.var["Lambda"].value(), ampl.obj["Upper"].value())

solved 0.0 0.0 0.0 2.0
